In [6]:
import joblib
import json
import numpy as np
import pandas as pd

In [16]:
modelo_completo = joblib.load('../../Limpieza/data/models/modelo_completo_datos_base.joblib')
with open('características.json', 'r') as f:
    caracteristicas = json.load(f)

In [17]:
modelo = modelo_completo['modelo']
scaler = modelo_completo['scaler']
mean = modelo_completo['mean']
cov = modelo_completo['cov']
columnas = modelo_completo['columnas']

In [18]:
# 2. Función para generar datos de prueba
def generar_datos_prueba(n_muestras=1):
    # Generar datos aleatorios
    datos = np.random.multivariate_normal(mean, cov, n_muestras)
    datos_escalados = scaler.inverse_transform(datos)
    df_prueba = pd.DataFrame(datos_escalados, columns=columnas)

    # Hacer predicción
    prediccion = modelo.predict(df_prueba)

    # Añadir predicción al dataframe
    df_prueba['Prediccion_Accesos'] = prediccion

    return df_prueba


In [19]:
# 3. Generar y mostrar algunos datos de prueba
datos_prueba = generar_datos_prueba(5)
print("\nDatos de prueba generados:")
print(datos_prueba)


Datos de prueba generados:
           AÑO  TRIMESTRE  VELOCIDAD BAJADA  VELOCIDAD SUBIDA   Latitud  \
0  2022.371585   1.757638          0.169803          0.325228  0.469092   
1  2023.340941   0.573703          3.202696          1.997802 -1.139597   
2  2023.859635   1.993315         -1.754342         -0.825355  1.314947   
3  2023.432142   3.473308          0.639849          0.525918  1.142698   
4  2021.848419   4.157903          3.680541          3.606598  1.211852   

   Longitud  Tasa_Crecimiento  Densidad_Accesos  Promedio_Movil  \
0 -1.181173        364.055480          0.680564      266.039825   
1  0.748606        -24.535378         -0.050861       18.174662   
2  2.200413        -45.485599         -0.082743       71.389340   
3  0.386589       -262.918635         -0.466023      -38.316267   
4  0.712374       -213.059301         -0.677268      -73.458200   

   Indice_Velocidad  Prediccion_Accesos  
0          0.247949              504.81  
1          2.630624               

In [20]:
# 4. Función para hacer predicciones con datos específicos
def predecir_accesos(datos_entrada):
    """
    Hacer predicciones usando el modelo cargado.

    Args:
        datos_entrada (dict): Diccionario con los valores de las características

    Returns:
        float: Predicción de accesos a internet
    """
    # Crear DataFrame con las columnas necesarias
    df_entrada = pd.DataFrame([datos_entrada])

    # Asegurar que tenemos todas las columnas necesarias
    for col in columnas:
        if col not in df_entrada.columns:
            df_entrada[col] = 0

    # Reordenar columnas como el modelo espera
    df_entrada = df_entrada[columnas]

    # Hacer predicción
    prediccion = modelo.predict(df_entrada)
    return prediccion[0]

In [21]:
datos_ejemplo = {
    'AÑO': 2024,
    'TRIMESTRE': 1,
    'VELOCIDAD BAJADA': 100,
    'VELOCIDAD SUBIDA': 50,
    'Latitud': 4.5,
    'Longitud': -74.2,
    'Tasa_Crecimiento': 0.1,
    'Densidad_Accesos': 100,
    'Promedio_Movil': 150,
    'Indice_Velocidad': 75
}

prediccion = predecir_accesos(datos_ejemplo)
print(f"\nPredicción para los datos de ejemplo: {prediccion:.2f} accesos")


Predicción para los datos de ejemplo: 10554.03 accesos


In [22]:
# 6. Función para evaluar la calidad de las predicciones
def evaluar_prediccion(prediccion):
    """
    Evaluar si una predicción es realista.
    """
    if prediccion < 0:
        return "Predicción no válida: valor negativo"
    elif prediccion > 1000000:
        return "Predicción no válida: valor muy alto"
    else:
        return "Predicción válida"

print("\nEvaluación de la predicción:", evaluar_prediccion(prediccion))


Evaluación de la predicción: Predicción válida
